<a href="https://colab.research.google.com/github/ronsong1234/Grand_IDC_Live/blob/main/notebooks/grandqc_slide_quality_with_idc_CLEAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Slide Quality Control with GrandQC and NCI Imaging Data Commons

Automated quality control (QC) is an essential preprocessing step in computational pathology pipelines. Artifacts such as tissue folds, out-of-focus regions, pen markings, and air bubbles can introduce noise that degrades the performance of downstream AI models.

[**GrandQC**](https://github.com/cpath-ukk/grandqc) is an open-source tool for automated tissue detection and multi-class artifact segmentation in whole slide images (WSIs). It was validated across slides from 19 international pathology departments and published in:

> Weng Z., Seper A., Pryalukhin A., et al. *GrandQC: A comprehensive solution to quality control problem in digital pathology.* Nature Communications 15, 10685 (2024). https://doi.org/10.1038/s41467-024-54769-y

[**NCI Imaging Data Commons (IDC)**](https://portal.imaging.datacommons.cancer.gov/) hosts ~100 TB of cancer imaging data, including thousands of H&E-stained whole slide images from TCGA, CPTAC, HTAN, and other programs — all stored as DICOM and freely accessible without authentication.

This notebook demonstrates how to:
1. **Discover** H&E-stained whole slide images in IDC using `idc-index`
2. **Set up** the corrected GrandQC-x-IDC pipeline and download its pre-trained models
3. **Download** slides from IDC and run GrandQC directly on DICOM files
4. **Visualize** tissue detection and artifact segmentation results
5. **Validate** the DICOM-based pipeline against GrandQC's pre-computed TCGA QC masks

### What changed in this (corrected) version

This notebook runs inference through the direct-DICOM pipeline in `modules/grandqc_qc.py`, which includes two fixes:

- **Task 1 — Boundary-anchored edge tiling.** At the right/bottom slide edges, the old code read a *partial* patch and stretched that thin strip up to the model's 512×512 input, distorting edge tiles so the model labeled them as background (large blocky artifacts, worse at higher resolution). The corrected tiler shifts the window backwards to anchor at `width − patch` / `height − patch`, so a **complete, unpadded** 512×512 block always reaches the model, then realigns the prediction to the grid.
- **Task 2 — `colorize_mask` utility.** Raw QC masks store integer class ids 0–7, which look black in ordinary viewers. `colorize_mask` maps them to the GrandQC palette for inspection.

> ⚠️ **Runtime requirement**: Run on a **T4 high-RAM** runtime. In Colab, select **Runtime → Change runtime type → T4 GPU** and enable **High-RAM**. The artifact step needs the GPU for reasonable speed (~30–45 s/slide vs. 10–30 min/slide on CPU); high-RAM avoids OOM on larger slides.


## Disclaimer

The code and data of this repository are provided to promote reproducible research. They are not intended for clinical care or commercial use.

The software is provided "as is", without warranty of any kind, express or implied, including but not limited to the warranties of merchantability, fitness for a particular purpose and noninfringement. In no event shall the authors or copyright holders be liable for any claim, damages or other liability, whether in an action of contract, tort or otherwise, arising from, out of or in connection with the software or the use or other dealings in the software.

**GrandQC license**: Creative Commons Attribution-NonCommercial-ShareAlike 4.0 (CC BY-NC-SA 4.0). Non-commercial research use only; citation of the GrandQC paper is required.

## Part 1: Discover H&E Slides in IDC

IDC provides the `idc-index` Python package for querying slide metadata and downloading DICOM files. We start by installing it and exploring the available whole slide image collections.

In [ ]:
%%capture
# IDC access + OpenSlide (used only for the DICOM read sanity-check in Part 3).
!pip install --upgrade idc-index openslide-python openslide-bin
# Dependencies for the direct-DICOM GrandQC pipeline in modules/grandqc_qc.py.
# segmentation-models-pytorch 0.3.1 is pinned because model.predict() was removed
# in 0.4.0 and GrandQC relies on it. wsidicom streams IDC DICOM WSI tiles.
!pip install -q \
    "segmentation-models-pytorch==0.3.1" \
    "wsidicom>=0.20" \
    "pydicom" \
    "tifffile" \
    "imagecodecs" 

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image
from IPython.display import IFrame, display
from idc_index import IDCClient

idc_client = IDCClient()
print(f"IDC data version: {idc_client.get_idc_version()}")

### 1.1 Overview of Slide Microscopy Collections

IDC stores whole slide images using the DICOM Slide Microscopy (SM) standard. The `sm_index` table extends the main metadata index with pathology-specific attributes including staining protocol, tissue type, pixel spacing, and image dimensions.

In [ ]:
# Load the slide microscopy index
idc_client.fetch_index("sm_index")

# Summary of all SM collections
sm_collections = idc_client.sql_query("""
    SELECT
        i.collection_id,
        COUNT(DISTINCT i.PatientID)      AS patients,
        COUNT(DISTINCT i.SeriesInstanceUID) AS series,
        ROUND(SUM(i.series_size_MB) / 1024.0, 1) AS size_GB,
        i.license_short_name
    FROM index i
    WHERE i.Modality = 'SM'
    GROUP BY i.collection_id, i.license_short_name
    ORDER BY patients DESC
""")

print(f"Total SM collections: {len(sm_collections)}")
print(f"Total SM series:      {sm_collections['series'].sum():,}")
print(f"Total size:           {sm_collections['size_GB'].sum():.0f} GB")
print()
sm_collections.head(15)

### 1.2 Filter for H&E-Stained Slides

The `sm_index` table includes a `staining_usingSubstance_CodeMeaning` column (stored as an array) that records the staining protocol for each slide. We filter for slides stained with hematoxylin, which identifies H&E preparations.

In [ ]:
# Count H&E slides per collection
he_by_collection = idc_client.sql_query("""
    SELECT
        i.collection_id,
        COUNT(DISTINCT i.PatientID)         AS patients,
        COUNT(DISTINCT i.SeriesInstanceUID) AS he_series,
        ROUND(SUM(i.series_size_MB) / 1024.0, 1) AS size_GB
    FROM index i
    JOIN sm_index s ON i.SeriesInstanceUID = s.SeriesInstanceUID
    WHERE array_to_string(s.staining_usingSubstance_CodeMeaning, ', ') LIKE '%hematoxylin%'
    GROUP BY i.collection_id
    ORDER BY he_series DESC
""")

print(f"Collections with H&E slides: {len(he_by_collection)}")
print(f"Total H&E series: {he_by_collection['he_series'].sum():,}")
he_by_collection.head(15)

### 1.3 Select Slides for Quality Control

For this tutorial we work with slides from the [TCGA-BRCA](https://portal.imaging.datacommons.cancer.gov/explore/filters/?collection_id=tcga_brca) collection (The Cancer Genome Atlas — Breast Cancer). This collection is a good choice because:

- It contains over 3,000 H&E-stained slides
- GrandQC provides **pre-computed QC masks for the TCGA cohorts**, which we use to validate our DICOM-based run in Part 6

Those reference masks cover only the **diagnostic FFPE slides** (TCGA `DX` barcodes), not the frozen-section slides (`TS`/`BS`), so we deliberately select diagnostic slides here. We query the `sm_index` to retrieve per-slide metadata including the `ContainerIdentifier` (the original TCGA slide barcode), which we use to match IDC series against the pre-computed GrandQC masks.

In [ ]:
# Query TCGA-BRCA H&E slides with key metadata
tcga_brca_he = idc_client.sql_query("""
    SELECT
        i.SeriesInstanceUID,
        i.PatientID,
        s.ContainerIdentifier,
        s.primaryAnatomicStructureModifier_CodeMeaning AS tissue_type,
        s.max_TotalPixelMatrixColumns                  AS width_px,
        s.max_TotalPixelMatrixRows                     AS height_px,
        s.min_PixelSpacing_2sf                         AS pixel_spacing_mm,
        s.ObjectiveLensPower                           AS objective_power,
        ROUND(i.series_size_MB, 1)                     AS size_MB
    FROM index i
    JOIN sm_index s ON i.SeriesInstanceUID = s.SeriesInstanceUID
    WHERE i.collection_id = 'tcga_brca'
      AND array_to_string(s.staining_usingSubstance_CodeMeaning, ', ') LIKE '%hematoxylin%'
    ORDER BY i.series_size_MB ASC
""")

print(f"TCGA-BRCA H&E slides: {len(tcga_brca_he)}")
print(f"Tissue types: {tcga_brca_he['tissue_type'].value_counts().to_dict()}")
print(f"\nSize range: {tcga_brca_he['size_MB'].min()} – {tcga_brca_he['size_MB'].max()} MB")
tcga_brca_he.head(10)

### 1.4 Select a Representative Set of Slides

For the hands-on GrandQC run we use a small set of **diagnostic (DX) H&E slides** that:
- are confirmed to have pre-computed GrandQC reference masks in the Zenodo `BRCA.tar` archive (so Part 6 can validate against them),
- span both magnifications GrandQC encounters in TCGA-BRCA (20× and 40×), and
- are small enough (≤ ~150 MB) for quick download and processing in a notebook.

All TCGA-BRCA diagnostic slides are primary-tumor sections, so the variety here comes from magnification rather than tissue type. The five barcodes below were verified against the Zenodo archive.

In [ ]:
# Diagnostic (DX) H&E slides verified to have pre-computed GrandQC reference
# masks in the TCGA-BRCA archive on Zenodo (record 14041578). Every DX slide is
# primary tumor, so the variety here is in objective power (40x and 20x).
DEMO_BARCODES = [
    "TCGA-AC-A23G-01Z-00-DX1",  # 40x,  44 MB
    "TCGA-AC-A23C-01Z-00-DX1",  # 40x,  57 MB
    "TCGA-AC-A62V-01Z-00-DX1",  # 40x,  60 MB
    "TCGA-MS-A51U-01Z-00-DX1",  # 20x,  76 MB
    "TCGA-A8-A0AB-01Z-00-DX1",  # 20x, 151 MB
]

available = set(tcga_brca_he["ContainerIdentifier"])
present   = [b for b in DEMO_BARCODES if b in available]
missing   = [b for b in DEMO_BARCODES if b not in available]
if missing:
    print(f"WARNING: {len(missing)} barcode(s) not found in this IDC version: {missing}")

demo_slides = (
    tcga_brca_he[tcga_brca_he["ContainerIdentifier"].isin(present)]
    .drop_duplicates(subset="ContainerIdentifier")
    .set_index("ContainerIdentifier")
    .loc[present]                  # preserve our chosen order
    .reset_index()
)

print(f"Selected {len(demo_slides)} slides for GrandQC processing:")
display(
    demo_slides[[
        "PatientID", "ContainerIdentifier", "tissue_type",
        "objective_power", "width_px", "height_px", "size_MB"
    ]]
)

In [ ]:
# Preview slides in the IDC SLIM viewer (pathology viewer)
for _, row in demo_slides.iterrows():
    url = idc_client.get_viewer_URL(seriesInstanceUID=row["SeriesInstanceUID"])
    print(f"{row['ContainerIdentifier']}  ({row['tissue_type']}, {row['size_MB']} MB)")
    print(f"  {url}")

## Part 2: Set Up the Corrected GrandQC-x-IDC Pipeline

Instead of shelling out to GrandQC's stock inference scripts, this notebook uses the thin, direct-DICOM pipeline in `modules/grandqc_qc.py`. It runs GrandQC's official tissue-detection and artifact models (same weights, downloaded from Zenodo) but reads IDC DICOM series directly through `wsidicom` and applies the **boundary-anchored edge-tiling fix**.

Steps:
1. Make `modules.grandqc_qc` importable (clone the repo in Colab; already present when run from the repo).
2. Confirm the tiling fix and `colorize_mask` utility are available.
3. Download the pre-trained models from Zenodo to the paths the pipeline expects.


### 2.1 Import the Pipeline

In [ ]:
import os, sys, subprocess

# In Colab, clone the repo that contains the modules/ package. When this notebook
# runs from inside the repository, the import simply succeeds.
try:
    import modules.grandqc_qc  # noqa: F401
except ModuleNotFoundError:
    REPO_URL = "https://github.com/ronsong1234/Grand_IDC_Live.git"
    REPO_DIR = "Grand_IDC_Live"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    sys.path.insert(0, os.path.abspath(REPO_DIR))
    import modules.grandqc_qc  # noqa: F401

from modules.grandqc_qc import (
    run_grandqc, colorize_mask, download_weights, CLASS_NAMES, CLASS_COLORS,
)
print("Imported corrected GrandQC pipeline from modules.grandqc_qc")
print("Classes:", CLASS_NAMES)

### 2.2 The Boundary-Anchored Edge-Tiling Fix (Task 1)

The pipeline tiles each slide into `native_patch × native_patch` windows (the level-0 footprint of the model's 512×512 input at the artifact MPP) and, for tissue detection, into 512×512 windows on the MPP-10 thumbnail.

**Before (broken):** edge tiles read only the available strip (`min(native_patch, width − x)`) and resized it up to 512×512 — stretching, e.g., a 40 px strip across the full input. The model saw distorted tissue and labeled it background, producing large blocky edge artifacts (more visible on higher-resolution slides, which have larger patches).

**After (fixed):** when a window would cross the right/bottom edge, its origin is shifted **backwards** to anchor at `width − native_patch` / `height − native_patch`, so a **complete, unpadded** block is read and resized 1:1. The prediction is then realigned to the grid cell (`_predict_artifact_patch` / `_anchored_start`), and any strip that falls outside the slide is set to the background class. No top-left padding is used.

The cell below confirms the corrected helpers are present.


In [ ]:
from modules import grandqc_qc as gq

# Confirm Task 1 (boundary-anchored tiling) and Task 2 (colorize_mask) are present.
checks = {
    "boundary-anchored artifact tiling (_predict_artifact_patch)": hasattr(gq, "_predict_artifact_patch"),
    "edge-anchor helper (_anchored_start)":                        hasattr(gq, "_anchored_start"),
    "colorize_mask utility":                                       hasattr(gq, "colorize_mask"),
}
for name, ok in checks.items():
    print(f"  [{'OK ' if ok else 'MISSING'}] {name}")
assert all(checks.values()), "Corrected pipeline helpers are missing — check the modules/ package."

# Download the official GrandQC weights (tissue MPP10 + artifact MPP1.5) from Zenodo
# to the paths modules.grandqc_qc expects. Existing non-empty files are skipped.
print("\nDownloading / verifying GrandQC weights ...")
weight_paths = download_weights(artifact_mpp=1.5)
for name, p in weight_paths.items():
    size_mb = os.path.getsize(p) / 1024**2
    print(f"  {name}: {p}  ({size_mb:.1f} MB)")

## Part 3: Download Slides from IDC

`idc_client.download_from_selection()` fetches DICOM files for the requested series and places each series in its own subdirectory named by `SeriesInstanceUID`.

After downloading we rename each subdirectory to its `ContainerIdentifier` (the original TCGA slide barcode). This ensures that GrandQC's output files and the pre-computed TCGA mask filenames are consistently keyed by the human-readable barcode rather than an opaque UID.

### 3.1 Download DICOM Series

In [ ]:
SLIDE_DIR = "slides"
os.makedirs(SLIDE_DIR, exist_ok=True)

print(f"Downloading {len(demo_slides)} slides  →  {SLIDE_DIR}/")
print("(This may take a few minutes depending on file sizes and network speed)\n")

idc_client.download_from_selection(
    downloadDir=SLIDE_DIR,
    seriesInstanceUID=demo_slides["SeriesInstanceUID"].tolist(),
    dirTemplate="%SeriesInstanceUID",
)
print("\nDownload complete.")

### 3.2 Rename Series Directories to TCGA Barcodes

GrandQC uses the directory/file name as the key for all output files (tissue mask, QC map, overlay, TSV report row). By renaming each downloaded series directory from its `SeriesInstanceUID` to its `ContainerIdentifier` (TCGA barcode), we get:

- Human-readable output filenames (`TCGA-AC-A23G-01Z-00-DX1_mask.png` instead of a UID)
- Straightforward matching with GrandQC's pre-computed TCGA masks (Part 6), which embed the same barcode in their filename

In [ ]:
import shutil

# Build SeriesInstanceUID → ContainerIdentifier map
uid_to_barcode = dict(zip(demo_slides["SeriesInstanceUID"], demo_slides["ContainerIdentifier"]))

# Rename each downloaded UID directory to its barcode. This is written to be
# safe to re-run, and to tolerate the download cell being re-run (which would
# re-create the UID directory next to an already-renamed barcode directory).
for uid, barcode in uid_to_barcode.items():
    src = os.path.join(SLIDE_DIR, uid)      # freshly downloaded, named by UID
    dst = os.path.join(SLIDE_DIR, barcode)  # renamed target, named by barcode

    if os.path.isdir(src) and os.path.isdir(dst):
        # Both present → the download cell was re-run after a previous rename.
        # The UID directory is a redundant copy of the same (immutable) series,
        # so drop it and keep the barcode directory downstream code expects.
        shutil.rmtree(src)
        print(f"  Removed duplicate UID dir; kept {barcode}")
    elif os.path.isdir(src):
        os.rename(src, dst)
        print(f"  {uid}  →  {barcode}")
    elif os.path.isdir(dst):
        print(f"  Already renamed: {barcode}")
    else:
        print(f"  WARNING: directory not found: {uid}")

print("\nSlide directories after renaming:")
for d in sorted(os.listdir(SLIDE_DIR)):
    dpath = os.path.join(SLIDE_DIR, d)
    if os.path.isdir(dpath):
        n = sum(1 for f in os.listdir(dpath) if f.lower().endswith(".dcm"))
        print(f"  {d}/  ({n} .dcm files)")

### 3.3 Verify OpenSlide Can Read the DICOM Files

OpenSlide 4.0 (bundled in `openslide-bin`) reads DICOM WSI files natively. Opening any one `.dcm` file from a series directory is sufficient — OpenSlide discovers the remaining pyramid levels from the other files in the same directory that share the same `SeriesInstanceUID`.

In [ ]:
import openslide

print(f"OpenSlide version: {openslide.__library_version__}\n")

for barcode in uid_to_barcode.values():
    slide_dir = os.path.join(SLIDE_DIR, barcode)
    dcm_files = sorted(f for f in os.listdir(slide_dir) if f.lower().endswith(".dcm"))
    if not dcm_files:
        print(f"  {barcode}: no .dcm files found")
        continue
    entry = os.path.join(slide_dir, dcm_files[0])
    try:
        slide = openslide.OpenSlide(entry)
        w, h = slide.level_dimensions[0]
        mpp = slide.properties.get("openslide.mpp-x", "N/A")
        vendor = slide.properties.get("openslide.vendor", "N/A")
        print(f"  {barcode}")
        print(f"    {w:,} × {h:,} px  |  MPP={mpp}  |  vendor={vendor}  |  levels={slide.level_count}")
        slide.close()
    except Exception as exc:
        print(f"  {barcode}: ERROR — {exc}")

print("\nAll slides verified.")

## Part 4: Run GrandQC Quality Control

`run_grandqc()` performs both stages internally for each DICOM series:

| Stage | MPP | What it does |
|-------|-----|--------------|
| Tissue detection | 10 | Binary tissue vs. background mask (used to skip background patches) |
| Artifact segmentation | 1.5 | 7-class artifact map at 512×512 tiles, with **boundary-anchored edge tiling** |

The 7 classes are: **1** clean tissue · **2** folds · **3** dark spots · **4** pen marks · **5** air bubbles/edges · **6** out-of-focus · **7** background.

For each slide it writes a raw integer mask to `qc_output/mask_qc/<barcode>_mask.png`, per-tile scores to `qc_output/tiles/`, and a tidy per-slide summary (returned below and saved to `qc_output/grandqc_summary.csv`).

> **GPU note**: ~30–45 s/slide on a T4 GPU; 10–30 min/slide on CPU.


In [ ]:
from modules.grandqc_qc import run_grandqc

OUTPUT_DIR = "qc_output"
barcodes    = list(uid_to_barcode.values())
slide_paths = [os.path.join(SLIDE_DIR, b) for b in barcodes]

summary = run_grandqc(
    slide_paths,
    input_type="dicom",          # read IDC DICOM series directly via wsidicom
    artifact_mpp=1.5,            # artifact model resolution
    output_dir=OUTPUT_DIR,
    slide_ids=barcodes,          # key outputs by TCGA barcode
)

cols = ["slide_id", "width_px", "height_px", "mpp_x_um",
        "tissue_percentage", "artifact_percentage_of_tissue", "usable"]
print("Per-slide GrandQC summary:")
display(summary[cols].round(4))
print(f"\nRaw integer masks written to {OUTPUT_DIR}/mask_qc/  (one *_mask.png per slide)")

## Part 5: Visualize and Analyze Results

The pipeline stores raw **integer** masks (class ids 1–7), which render as near-black in ordinary viewers. We use the `colorize_mask` utility (Task 2) to map them to GrandQC's palette, then compute per-slide artifact fractions from the summary table.


### 5.1 Colorized Artifact Masks

`colorize_mask` maps each integer class to a distinct RGB color:

| Class | Color | Meaning |
|-------|-------|---------|
| 1 | Gray `#808080` | Clean tissue |
| 2 | Orange-red `#FF6347` | Folds |
| 3 | Green `#00FF00` | Dark spots |
| 4 | Red `#FF0000` | Pen marks |
| 5 | Magenta `#FF00FF` | Air bubbles / edges |
| 6 | Indigo `#4B0082` | Out-of-focus |
| 7 | White `#FFFFFF` | Background |


In [ ]:
from modules.grandqc_qc import colorize_mask

mask_dir   = os.path.join(OUTPUT_DIR, "mask_qc")
mask_files = [(b, os.path.join(mask_dir, f"{b}_mask.png")) for b in barcodes]
mask_files = [(b, p) for b, p in mask_files if os.path.exists(p)]

ncols = 2 if len(mask_files) > 1 else 1
nrows = (len(mask_files) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 6 * nrows))
axes = np.array(axes).reshape(-1)

for ax, (barcode, path) in zip(axes, mask_files):
    ax.imshow(colorize_mask(path))          # integer labels -> RGB
    ax.set_title(barcode, fontsize=9)
    ax.axis("off")
for ax in axes[len(mask_files):]:
    ax.axis("off")

plt.suptitle("GrandQC artifact masks (colorized with colorize_mask)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 5.2 Artifact Fractions per Slide

The summary table already reports each class as a fraction of the **tissue area** (background excluded). We plot the five artifact classes (2–6) per slide — the standard metric from the GrandQC paper.


In [ ]:
artifact_ids = [2, 3, 4, 5, 6]
frac_df = summary.set_index("slide_id")[[f"{CLASS_NAMES[c]}_fraction" for c in artifact_ids]] * 100
frac_df.columns = [CLASS_NAMES[c] for c in artifact_ids]

print("Artifact class fractions (% of tissue area):")
display(frac_df.round(2))

bar_colors = [tuple(v / 255 for v in CLASS_COLORS[c]) for c in artifact_ids]
ax = frac_df.plot(kind="bar", stacked=True, color=bar_colors,
                  figsize=(max(6, 2.5 * len(frac_df)), 5), edgecolor="none")
ax.set_ylabel("% of tissue area")
ax.set_xlabel("")
ax.set_title("GrandQC Artifact Fractions per Slide", fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right", fontsize=8)
ax.legend(loc="upper right", fontsize=8, framealpha=0.8)
plt.tight_layout()
plt.show()

### 5.4 View Original Slides in IDC SLIM Viewer

The IDC SLIM viewer is a web-based pathology viewer for DICOM whole slide images. The links below open each slide at full resolution. The GeoJSON annotations produced by GrandQC (`geojson_qc/`) can be loaded into QuPath or other tools that accept GeoJSON overlays for interactive artifact review.

In [ ]:
from IPython.display import IFrame, display

print("IDC SLIM viewer links (open in browser for full-resolution viewing):\n")
for _, row in demo_slides.iterrows():
    url = idc_client.get_viewer_URL(seriesInstanceUID=row["SeriesInstanceUID"])
    print(f"  {row['ContainerIdentifier']}")
    print(f"    {url}\n")

# Embed the first slide inline (750 px height)
first_url = idc_client.get_viewer_URL(seriesInstanceUID=demo_slides.iloc[0]["SeriesInstanceUID"])
print(f"Embedding: {demo_slides.iloc[0]['ContainerIdentifier']}")
display(IFrame(first_url, width="100%", height=750))

## Part 6: Validating the Pipeline Against Pre-computed TCGA Masks

In Parts 3–5 we ran GrandQC locally on IDC DICOM slides using OpenSlide's native DICOM reader — a path the GrandQC authors did not test in their original publication. Before using the pipeline on new data, it is good practice to verify that our results are consistent with the authors' reference outputs.

The GrandQC authors published pre-computed QC masks for the TCGA cohorts on Zenodo:

> **GrandQC pre-computed masks**: [zenodo.org/records/14041578](https://zenodo.org/records/14041578)  
> One tar archive per TCGA cohort (a few large cohorts are split into two parts), ~18 GB in total. The TCGA-BRCA archive (`BRCA.tar`) alone is ~2 GB and holds masks for 1,105 **diagnostic (DX) slides**.

Each mask is a PNG named after the original slide file — `{barcode}.{UUID}.svs_mask.png` — with integer labels 1–7, the same classes `main.py` produces. Rather than download the full 2 GB cohort archive, we pull just the reference masks for our five demo slides (~700 KB, extracted from `BRCA.tar` and re-hosted) and compare artifact fractions against our local DICOM run.

### 6.1 Available Cohort Archives

The Zenodo record contains one tar file per TCGA cohort; a few large cohorts (LUAD, LUSC, THCA, TGCT) are split into two parts. The cell below lists them straight from the Zenodo API — no large download.

In [ ]:
import requests

# Fetch the Zenodo record metadata to get the authoritative file list
rec = requests.get("https://zenodo.org/api/records/14041578").json()
files = rec["files"]

rows_z = []
for f in sorted(files, key=lambda x: x["key"]):
    rows_z.append({
        "archive": f["key"],
        "size_MB": round(f["size"] / 1024**2, 0),
    })

zenodo_df = pd.DataFrame(rows_z)
total_gb  = zenodo_df["size_MB"].sum() / 1024
print(f"{len(zenodo_df)} archives  |  total: {total_gb:.1f} GB\n")
display(zenodo_df)

### 6.2 Download the Reference Masks for Our Demo Slides

We download just the pre-computed masks for the five demo slides (~700 KB), extracted from `BRCA.tar` and re-hosted as a small folder. Each reference mask is named `{barcode}.{UUID}.svs_mask.png`, whereas our local masks in `qc_output/mask_qc/` are keyed by the bare `ContainerIdentifier` barcode — so we also build a lookup from barcode to the full reference filename.

> To reproduce population-level statistics across the whole cohort, download the full `BRCA.tar` (~2 GB) from the Zenodo record above and point `REF_MASKS_DIR` at the extracted folder; the comparison loop below works unchanged on any number of masks.


In [ ]:
import io, re, zipfile, requests

REF_MASKS_DIR = "brca_ref_masks"
os.makedirs(REF_MASKS_DIR, exist_ok=True)

# Pre-computed GrandQC masks for our five demo slides, extracted from the
# TCGA-BRCA Zenodo archive (record 14041578) and re-hosted as a small folder
# (~700 KB) so we don't have to download the full ~2 GB BRCA.tar. The link is a
# Dropbox shared folder; appending '&dl=1' returns the whole folder as a zip.
REF_MASKS_URL = (
    "https://www.dropbox.com/scl/fo/z5jtldr5ed384s8nntiwe/"
    "ALUgs22f_0PI8oGO6XJ5eG0?rlkey=b92k8rt6jvqllwkad31obvbyx&dl=1"
)

resp = requests.get(REF_MASKS_URL)
resp.raise_for_status()
with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
    for member in zf.namelist():
        if member.endswith("_mask.png"):
            target = os.path.join(REF_MASKS_DIR, os.path.basename(member))
            with zf.open(member) as src, open(target, "wb") as dst:
                dst.write(src.read())

ref_masks = sorted(f for f in os.listdir(REF_MASKS_DIR) if f.endswith("_mask.png"))

# Reference masks are named either '{barcode}.{UUID}.svs_mask.png' or
# '{barcode}_mask.png'. Extract the bare TCGA ContainerIdentifier barcode so the
# comparison cannot silently miss files because of a suffix mismatch.
def extract_tcga_slide_id(name):
    match = re.search(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}-[A-Z0-9]{3}-[A-Z0-9]{2}-[A-Z0-9]{3})", os.path.basename(str(name)))
    if not match:
        raise ValueError(f"Could not extract TCGA slide id from {name}")
    return match.group(1)

ref_mask_by_barcode = {extract_tcga_slide_id(f): f for f in ref_masks}

print(f"Downloaded {len(ref_masks)} reference masks to {REF_MASKS_DIR}/")
for f in ref_masks:
    print(f"  {extract_tcga_slide_id(f)} -> {f}")

### 6.3 Pipeline Validation: Compare Local vs. Reference Masks

The reference masks in `brca_ref_masks/` were produced by the GrandQC authors. The **local** masks are the ones we just generated in `qc_output/mask_qc/` with the boundary-anchored pipeline. We compare per-class artifact fractions for each demo slide; close agreement indicates the direct-DICOM pipeline reproduces the authors' reference outputs.


In [ ]:
from pathlib import Path

CLASS_NAMES_SHORT = {1: "Clean tissue", 2: "Folds", 3: "Dark spots",
                     4: "Pen marks", 5: "Bubbles/edges", 6: "Out-of-focus"}

def mask_fractions(mask_path):
    m = np.array(Image.open(mask_path))
    if m.ndim == 3:
        m = m[..., 0]
    tissue_px = np.sum(m != 7)
    if tissue_px == 0:
        return None
    return {name: np.sum(m == cls) / tissue_px * 100
            for cls, name in CLASS_NAMES_SHORT.items()}

def local_mask_path(barcode):
    return Path(OUTPUT_DIR) / "mask_qc" / f"{barcode}_mask.png"

compare_rows = []
for barcode in barcodes:
    lp = local_mask_path(barcode)
    ref_fname = ref_mask_by_barcode.get(barcode)
    rp = Path(REF_MASKS_DIR) / ref_fname if ref_fname else None
    if not lp.exists():
        print(f"  MISSING local mask: {barcode} at {lp}")
        continue
    if rp is None or not rp.exists():
        print(f"  MISSING reference mask: {barcode}")
        continue
    lf, rf = mask_fractions(lp), mask_fractions(rp)
    if lf is None or rf is None:
        continue
    for name in CLASS_NAMES_SHORT.values():
        compare_rows.append({
            "slide": barcode, "class": name,
            "local_%": round(lf[name], 2),
            "ref_%":   round(rf[name], 2),
            "delta_pp": round(lf[name] - rf[name], 2),
        })

compare_df = pd.DataFrame(compare_rows)
print("Per-class fraction comparison (local DICOM run vs. Zenodo reference):\n")
display(compare_df.pivot_table(index="slide", columns="class",
                               values=["local_%", "ref_%", "delta_pp"]))

In [ ]:
from modules.grandqc_qc import CLASS_COLORS

mpl_colors   = [tuple(v / 255 for v in CLASS_COLORS[c]) for c in [1, 2, 3, 4, 5, 6]]
class_colors = {name: mpl_colors[i] for i, name in enumerate(CLASS_NAMES_SHORT.values())}

fig, ax = plt.subplots(figsize=(5, 5))
for cls_name, grp in compare_df.groupby("class"):
    ax.scatter(grp["ref_%"], grp["local_%"], label=cls_name,
               color=class_colors[cls_name], s=60,
               edgecolors="black", linewidths=0.4, zorder=3)

lim_max = max(compare_df[["local_%", "ref_%"]].max()) * 1.05
ax.plot([0, lim_max], [0, lim_max], "k--", linewidth=0.8, label="y = x")
ax.set_xlabel("Reference fraction (%) — Zenodo")
ax.set_ylabel("Local fraction (%) — DICOM run")
ax.set_title("Local vs. Reference Artifact Fractions", fontweight="bold", fontsize=9)
ax.legend(fontsize=7, framealpha=0.8)
ax.set_xlim(0, lim_max)
ax.set_ylim(0, lim_max)
plt.tight_layout()
plt.show()

### 6.4 Compare Reference and Local Results Side by Side

Per-class fractions (6.3) summarise agreement as numbers. Here we place the two artifact maps next to each other — the reference (Zenodo) result on the left and our local DICOM result on the right, one slide per row — so differences in *where* each artifact class is detected are easy to spot.

Both masks are colourised with the same GrandQC palette: clean tissue (grey) plus the five artifact classes; background (class 7) is shown white. The reference and local masks can differ slightly in pixel dimensions, but since the panels are shown side by side rather than overlaid, no resizing is needed.

In [ ]:
from matplotlib.patches import Patch

def _display_mask(path, pad=5):
    # Padding (class 0) and background (class 7) shown white, then crop to the
    # tissue bounding box so reference and local panels are framed identically
    # regardless of GrandQC's edge padding.
    m = np.array(Image.open(path))
    if m.ndim == 3:
        m = m[..., 0]
    rgb = colorize_mask(m)          # class 7 is already white in the palette
    rgb[m == 0] = 255               # class-0 padding -> white (drops the black L border)
    fg = ~np.isin(m, (0, 7))        # tissue + artifacts = foreground
    if fg.any():
        ys, xs = np.where(fg)
        y0, y1 = max(0, ys.min() - pad), min(rgb.shape[0], ys.max() + 1 + pad)
        x0, x1 = max(0, xs.min() - pad), min(rgb.shape[1], xs.max() + 1 + pad)
        rgb = rgb[y0:y1, x0:x1]
    return rgb

slides = barcodes
n = len(slides)
fig, axes = plt.subplots(n, 2, figsize=(13, 5.0 * n))
axes = np.atleast_2d(axes)

for row, barcode in enumerate(slides):
    lp = local_mask_path(barcode)
    ref_fname = ref_mask_by_barcode.get(barcode)
    rp = Path(REF_MASKS_DIR) / ref_fname if ref_fname else None

    ax_ref, ax_local = axes[row, 0], axes[row, 1]
    ax_ref.axis("off")
    ax_local.axis("off")

    if rp and rp.exists():
        ax_ref.imshow(_display_mask(rp))
    else:
        ax_ref.text(0.5, 0.5, "reference mask missing", ha="center", va="center")
    if lp.exists():
        ax_local.imshow(_display_mask(lp))
    else:
        ax_local.text(0.5, 0.5, "local mask missing", ha="center", va="center")

    ax_ref.set_title(f"{barcode}\nReference (Zenodo)", fontsize=10)
    ax_local.set_title("Local (corrected DICOM)", fontsize=10)

legend_handles = [Patch(facecolor=mpl_colors[i], edgecolor="black",
                        label=list(CLASS_NAMES_SHORT.values())[i]) for i in range(6)]
fig.legend(handles=legend_handles, loc="lower center", ncol=6, frameon=True)
fig.suptitle("Artifact maps — reference vs. corrected local DICOM run",
             fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()

### 6.5 Spatial Agreement: Per-Class Dice / IoU

The fraction comparison in 6.3 measures *how much* of each class is present, but two masks can have identical fractions while disagreeing on *where* the classes are. Here we compute per-class **Dice** and **IoU** — spatial overlap metrics — between the local and reference masks. Dice for class *c* is `2*|A n B| / (|A|+|B|)`; IoU is `|A n B| / |A u B|`.

> **Alignment caveat**: the reference masks retain GrandQC's edge padding (the black class-0 borders in 6.4), so the two masks differ slightly in extent. We resize the reference onto the local mask's grid (nearest-neighbour) before overlap, which is approximate near the padded borders but faithful over the tissue interior. A class absent in *both* masks yields `NaN` (undefined) and is excluded from averages; a class present in only one yields Dice 0.


In [ ]:
from PIL import Image

DICE_CLASS_IDS = (1, 2, 3, 4, 5, 6)   # clean tissue + five artifact classes

def _load_labels(path, target_shape=None):
    m = np.array(Image.open(path))
    if m.ndim == 3:
        m = m[..., 0]
    if target_shape is not None and m.shape != target_shape:
        # Nearest-neighbour keeps integer labels intact.
        m = np.array(Image.fromarray(m.astype(np.uint8)).resize(
            (target_shape[1], target_shape[0]), Image.Resampling.NEAREST))
    return m

def dice_iou_per_class(local_path, ref_path, class_ids=DICE_CLASS_IDS):
    local = _load_labels(local_path)
    ref = _load_labels(ref_path, target_shape=local.shape)   # align ref -> local grid
    out = {}
    for c in class_ids:
        a, b = (local == c), (ref == c)
        inter = int(np.logical_and(a, b).sum())
        union = int(np.logical_or(a, b).sum())
        sa, sb = int(a.sum()), int(b.sum())
        if sa + sb == 0:
            out[c] = (np.nan, np.nan)          # class absent in both -> undefined
        else:
            out[c] = (2 * inter / (sa + sb), (inter / union) if union else np.nan)
    return out

dice_rows = []
for barcode in barcodes:
    lp = local_mask_path(barcode)
    ref_fname = ref_mask_by_barcode.get(barcode)
    rp = Path(REF_MASKS_DIR) / ref_fname if ref_fname else None
    if not lp.exists() or rp is None or not rp.exists():
        continue
    for c, (dice, iou) in dice_iou_per_class(lp, rp).items():
        dice_rows.append({"slide": barcode, "class": CLASS_NAMES_SHORT[c],
                          "dice": dice, "iou": iou})

dice_df = pd.DataFrame(dice_rows)
print("Per-class Dice (spatial agreement, local vs. reference):")
display(dice_df.pivot_table(index="slide", columns="class", values="dice").round(3))
print("Per-class IoU:")
display(dice_df.pivot_table(index="slide", columns="class", values="iou").round(3))

### 6.6 Cohort-Scale Agreement Summary

Aggregate Dice/IoU across all compared slides, per class. Clean tissue (class 1) dominates the pixel count and will score highest; the artifact classes (2-6) are the discriminating signal. Rare classes on small tissue areas are inherently noisier, so read the per-class **slide count** alongside the means.

This summary generalizes unchanged to the full cohort: point `REF_MASKS_DIR` at the extracted `BRCA.tar` (~2 GB, 1,105 masks) and re-run Parts 6.2-6.6.


In [ ]:
summary_dice = (dice_df.groupby("class")
                .agg(slides=("dice", "count"),
                     mean_dice=("dice", "mean"),
                     median_dice=("dice", "median"),
                     mean_iou=("iou", "mean"))
                .round(3))
class_order = [CLASS_NAMES_SHORT[c] for c in DICE_CLASS_IDS]
summary_dice = summary_dice.reindex([c for c in class_order if c in summary_dice.index])

print(f"Cohort-scale agreement over {dice_df['slide'].nunique()} slide(s):")
display(summary_dice)

macro = dice_df["dice"].mean()
artifact_macro = dice_df[dice_df["class"] != "Clean tissue"]["dice"].mean()
print(f"Macro-average Dice - all classes:     {macro:.3f}")
print(f"Macro-average Dice - artifacts (2-6): {artifact_macro:.3f}")

# Mean Dice per class, colored by the GrandQC palette.
name_to_id = {v: k for k, v in CLASS_NAMES_SHORT.items()}
bar_colors = [tuple(v / 255 for v in CLASS_COLORS[name_to_id[c]]) for c in summary_dice.index]
ax = summary_dice["mean_dice"].plot(kind="bar", color=bar_colors, edgecolor="black", figsize=(7, 4))
ax.set_ylabel("Mean Dice")
ax.set_ylim(0, 1)
ax.set_xlabel("")
ax.set_title("Mean per-class Dice: local DICOM vs. reference", fontweight="bold", fontsize=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

### 6.7 Diagnosing Edge-Tile Differences (Color vs. Geometry)

Where the local and reference maps disagree at a boundary — e.g. the green (dark spots) blob at the right tip of TCGA-AC-A62V — it helps to know whether the cause is *geometry* (already ruled out: the boundary-anchored tiler places edge tiles 1:1) or *appearance*. The local pipeline decodes DICOM via `wsidicom`; the reference masks were produced from SVS via OpenSlide, and the artifact model is sensitive to colour/brightness.

The cell below reads the same right-edge tile through both readers and reports the per-channel pixel difference (`absdiff_wsidicom_vs_openslide`) and each reader's predicted classes. If the channel differences are more than a few units and class 3 (dark spots) appears for `wsidicom` but not `openslide`, the disagreement is a colour-decode difference, and the fix is to align the DICOM read's colour handling. If the differences are near zero, it is genuine model sensitivity on a small region (expected concordance).

> Requires OpenSlide (installed in Part 1) and reloads the artifact model.


In [ ]:
from modules.grandqc_qc import inspect_dicom_series, debug_compare_openslide_tile, MODEL_TILE_SIZE

A62V = "TCGA-AC-A62V-01Z-00-DX1"   # slide with the right-tip disagreement
src = os.path.join(SLIDE_DIR, A62V)
if not os.path.isdir(src):
    print(f"{A62V} not downloaded; pick one of the processed slides: {barcodes}")
else:
    info = inspect_dicom_series(src)
    W, H = int(info.loc[0, "width_px"]), int(info.loc[0, "height_px"])
    mpp = float(info.loc[0, "mpp_x_um"])
    ARTIFACT_MPP = 1.5
    native_patch = max(1, int(ARTIFACT_MPP / mpp * MODEL_TILE_SIZE))
    x0 = max(0, W - native_patch)   # right-edge column, boundary-anchored
    print(f"{A62V}: {W}x{H}px, mpp={mpp:.4f}, native_patch={native_patch}, anchored x0={x0}")
    for frac in (0.40, 0.50, 0.60):     # sweep the vertical band where the right tip sits
        y0 = min(max(0, int(H * frac) - native_patch // 2), max(0, H - native_patch))
        print()
        print(f"--- right-edge tile @ y0={y0} (frac={frac}) ---")
        df, imgs = debug_compare_openslide_tile(src, x_l0=x0, y_l0=y0,
                                                native_patch_size_px=native_patch,
                                                artifact_mpp=ARTIFACT_MPP)
        display(df)

### 6.8 Scaling to Cohort-level QC

Because each reference mask is just a small PNG, the comparison above scales to an entire cohort. Downloading the full `BRCA.tar` (~2 GB) from the Zenodo record gives pre-computed masks for all 1,105 TCGA-BRCA diagnostic slides — enough to compute population-level artifact statistics (medians, percentiles, distributions) **without running any inference locally**. We keep this notebook lightweight by validating on the five demo slides only; the loop in section 6.3 generalizes directly to the full mask set, reading one mask at a time so memory stays constant.

## Part 7: Cross-Cohort Generalization (TCGA-LUAD)

Parts 1-6 validated the pipeline on TCGA-BRCA (breast). BRCA is one tissue type and scanner mix, so strong agreement there does not by itself prove the pipeline generalizes. Here we repeat the validation on a **different cohort — TCGA-LUAD (lung adenocarcinoma)** — on 15 slides, using the same models, the same boundary-anchored tiling, and the same Dice/IoU metrics.

This part reuses functions defined earlier (`run_grandqc`, `colorize_mask`, `dice_iou_per_class`, `extract_tcga_slide_id`, `_display_mask`, `CLASS_NAMES_SHORT`, `DICE_CLASS_IDS`), so **run Parts 1-6 first**. To test yet another cohort, change the three `COHORT_*` variables in 7.1 (e.g. `prad` + `tcga_prad` + `PRAD.tar`).


### 7.1 Download and Extract the Cohort's Reference Masks

In [ ]:
import tarfile

# --- cohort parameters (change these to validate a different collection) ---
COHORT            = "luad"                 # short name used for output folders
COHORT_COLLECTION = f"tcga_{COHORT}"        # IDC collection id
COHORT_ARCHIVES   = ["LUAD.tar"]            # Zenodo archive(s); add "LUAD_2.tar" if <15 matches
N_COHORT          = 15
COHORT_REF_DIR    = f"{COHORT}_ref_masks"
os.makedirs(COHORT_REF_DIR, exist_ok=True)

rec = requests.get("https://zenodo.org/api/records/14041578").json()
for arc in COHORT_ARCHIVES:
    finfo = next(f for f in rec["files"] if f["key"] == arc)
    tar_path = arc
    if not os.path.exists(tar_path):
        size_mb = finfo["size"] / 1024**2
        print(f"Downloading {arc} ({size_mb:.0f} MB) ...")
        with requests.get(finfo["links"]["self"], stream=True) as r:
            r.raise_for_status()
            with open(tar_path, "wb") as out:
                for chunk in r.iter_content(1 << 20):
                    out.write(chunk)
    with tarfile.open(tar_path) as tar:
        members = [m for m in tar.getmembers() if m.name.endswith("_mask.png")]
        for m in members:
            with open(os.path.join(COHORT_REF_DIR, os.path.basename(m.name)), "wb") as f:
                f.write(tar.extractfile(m).read())
        print(f"  extracted {len(members)} masks from {arc}")

cohort_ref_by_barcode = {}
for f in sorted(os.listdir(COHORT_REF_DIR)):
    if not f.endswith("_mask.png"):
        continue
    bc = extract_tcga_slide_id(f)     # defined in Part 6.2
    if bc:
        cohort_ref_by_barcode[bc] = f
print(f"{len(cohort_ref_by_barcode)} reference masks available for {COHORT.upper()}")

### 7.2 Select 15 Slides Present in Both IDC and the Archive

In [ ]:
cohort_he = idc_client.sql_query(f"""
    SELECT
        i.SeriesInstanceUID,
        s.ContainerIdentifier,
        s.max_TotalPixelMatrixColumns AS width_px,
        s.max_TotalPixelMatrixRows    AS height_px,
        ROUND(i.series_size_MB, 1)    AS size_MB
    FROM index i
    JOIN sm_index s ON i.SeriesInstanceUID = s.SeriesInstanceUID
    WHERE i.collection_id = '{COHORT_COLLECTION}'
      AND array_to_string(s.staining_usingSubstance_CodeMeaning, ', ') LIKE '%hematoxylin%'
      AND s.ContainerIdentifier LIKE '%-DX%'
    ORDER BY i.series_size_MB ASC
""")

# keep only slides that also have a reference mask, dedup, take the N smallest
cohort_he = cohort_he.drop_duplicates(subset="ContainerIdentifier")
cohort_he = cohort_he[cohort_he["ContainerIdentifier"].isin(cohort_ref_by_barcode)]
cohort_slides = cohort_he.head(N_COHORT).reset_index(drop=True)

print(f"{len(cohort_slides)} {COHORT.upper()} slides present in both IDC and the reference archive "
      f"(requested {N_COHORT}):")
display(cohort_slides[["ContainerIdentifier", "width_px", "height_px", "size_MB"]])
if len(cohort_slides) < N_COHORT:
    print("Fewer than requested; add the cohort's _2 archive to COHORT_ARCHIVES for a larger pool.")

### 7.3 Download the DICOM Series and Key Them by Barcode

In [ ]:
COHORT_SLIDE_DIR = f"{COHORT}_slides"
os.makedirs(COHORT_SLIDE_DIR, exist_ok=True)

idc_client.download_from_selection(
    downloadDir=COHORT_SLIDE_DIR,
    seriesInstanceUID=cohort_slides["SeriesInstanceUID"].tolist(),
    dirTemplate="%SeriesInstanceUID",
)

cohort_uid_to_barcode = dict(zip(cohort_slides["SeriesInstanceUID"], cohort_slides["ContainerIdentifier"]))
for uid, barcode in cohort_uid_to_barcode.items():
    srcd = os.path.join(COHORT_SLIDE_DIR, uid)
    dstd = os.path.join(COHORT_SLIDE_DIR, barcode)
    if os.path.isdir(srcd) and os.path.isdir(dstd):
        shutil.rmtree(srcd)
    elif os.path.isdir(srcd):
        os.rename(srcd, dstd)
print(f"Downloaded and renamed {len(cohort_uid_to_barcode)} {COHORT.upper()} slides -> {COHORT_SLIDE_DIR}/")

### 7.4 Run GrandQC on the Cohort

In [ ]:
COHORT_OUTPUT_DIR = f"{COHORT}_qc_output"
cohort_barcodes = list(cohort_uid_to_barcode.values())
cohort_paths = [os.path.join(COHORT_SLIDE_DIR, b) for b in cohort_barcodes]

cohort_summary = run_grandqc(
    cohort_paths,
    input_type="dicom",
    artifact_mpp=1.5,
    output_dir=COHORT_OUTPUT_DIR,
    slide_ids=cohort_barcodes,
)
display(cohort_summary[["slide_id", "width_px", "height_px", "mpp_x_um",
                        "tissue_percentage", "artifact_percentage_of_tissue", "usable"]].round(4))

### 7.5 Per-Class Dice / IoU and Cohort Summary

In [ ]:
def cohort_local_mask_path(barcode):
    return Path(COHORT_OUTPUT_DIR) / "mask_qc" / f"{barcode}_mask.png"

cohort_dice_rows = []
for barcode in cohort_barcodes:
    lp = cohort_local_mask_path(barcode)
    rfn = cohort_ref_by_barcode.get(barcode)
    rp = Path(COHORT_REF_DIR) / rfn if rfn else None
    if not lp.exists() or rp is None or not rp.exists():
        continue
    for c, (dice, iou) in dice_iou_per_class(lp, rp).items():
        cohort_dice_rows.append({"slide": barcode, "class": CLASS_NAMES_SHORT[c],
                                 "dice": dice, "iou": iou})

cohort_dice_df = pd.DataFrame(cohort_dice_rows)
print(f"{COHORT.upper()} per-class Dice:")
display(cohort_dice_df.pivot_table(index="slide", columns="class", values="dice").round(3))

cohort_sum = (cohort_dice_df.groupby("class")
              .agg(slides=("dice", "count"), mean_dice=("dice", "mean"),
                   median_dice=("dice", "median"), mean_iou=("iou", "mean"))
              .round(3))
order = [CLASS_NAMES_SHORT[c] for c in DICE_CLASS_IDS]
cohort_sum = cohort_sum.reindex([c for c in order if c in cohort_sum.index])
print(f"{COHORT.upper()} agreement over {cohort_dice_df['slide'].nunique()} slides:")
display(cohort_sum)
print(f"Macro Dice - all classes:     {cohort_dice_df['dice'].mean():.3f}")
print(f"Macro Dice - artifacts (2-6): {cohort_dice_df[cohort_dice_df['class'] != 'Clean tissue']['dice'].mean():.3f}")

# Mean Dice per class vs. the BRCA run (Part 6.6), if that summary is available.
name_to_id = {v: k for k, v in CLASS_NAMES_SHORT.items()}
bar_colors = [tuple(v / 255 for v in CLASS_COLORS[name_to_id[c]]) for c in cohort_sum.index]
ax = cohort_sum["mean_dice"].plot(kind="bar", color=bar_colors, edgecolor="black", figsize=(7, 4))
ax.set_ylabel("Mean Dice")
ax.set_ylim(0, 1)
ax.set_xlabel("")
ax.set_title(f"{COHORT.upper()} mean per-class Dice: local DICOM vs. reference",
             fontweight="bold", fontsize=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

### 7.6 Reference vs. Local Maps (all cohort slides)

In [ ]:
from matplotlib.patches import Patch

show = cohort_barcodes
fig, axes = plt.subplots(len(show), 2, figsize=(13, 5.0 * len(show)))
axes = np.atleast_2d(axes)
for row, barcode in enumerate(show):
    lp = cohort_local_mask_path(barcode)
    rfn = cohort_ref_by_barcode.get(barcode)
    rp = Path(COHORT_REF_DIR) / rfn if rfn else None
    ax_ref, ax_local = axes[row, 0], axes[row, 1]
    ax_ref.axis("off")
    ax_local.axis("off")
    if rp and rp.exists():
        ax_ref.imshow(_display_mask(rp))
    else:
        ax_ref.text(0.5, 0.5, "reference mask missing", ha="center", va="center")
    if lp.exists():
        ax_local.imshow(_display_mask(lp))
    else:
        ax_local.text(0.5, 0.5, "local mask missing", ha="center", va="center")
    ax_ref.set_title(f"{barcode}  (Reference, Zenodo)", fontsize=9)
    ax_local.set_title("Local (corrected DICOM)", fontsize=9)
legend_handles = [Patch(facecolor=mpl_colors[i], edgecolor="black",
                        label=list(CLASS_NAMES_SHORT.values())[i]) for i in range(6)]
fig.legend(handles=legend_handles, loc="lower center", ncol=6, frameon=True)
fig.suptitle(f"{COHORT.upper()} - reference vs. corrected local DICOM run",
             fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()